# Late fusion: DeepCDR and MOLI, with and without proteomics, with and without sex-aware components
*Using sex-stratification and multi-omics data to predict drug response (TNAM)*

This notebook trains **DeepCDR** and **MOLI** as **late-fusion** models. Each omics layer and the drug get their own encoder, and the embeddings are concatenated only at the end. It covers the full 2 × 2 design:

| | no sex-aware | sex-aware |
|---|---|---|
| **expression + mutation + CNV** | `LF_<family>_noprot_base` | `LF_<family>_noprot_sexaware` |
| **+ proteomics** | `LF_<family>_prot_base` | `LF_<family>_prot_sexaware` |

for `<family>` ∈ {DeepCDR, MOLI}: 8 runs per seed per split.

| family | omics encoders | drug encoder | fusion | head | extra loss |
|---|---|---|---|---|---|
| `LF_DeepCDR` | DeepCDR dense branches (proteomics gets its own dense branch) + mutation 1-D CNN | DeepCDR GCN | concatenate `[omics..., drug]` (as in DeepCDR) | DeepCDR conv head | – |
| `LF_MOLI` | MOLI `Linear-BN-ReLU-Dropout` per omic | GCN | concatenate omics → L2-normalise (MOLI), concat L2-normalised drug | MLP | MOLI triplet |

**H3 (proteomics):** every run uses the *same* cell lines (lines with measured proteomics) and the same test rows, so "+ proteomics" vs "no proteomics" is a paired comparison on identical observations, as `docs/eval_plan.md` requires. **H4 (sex-aware):** sex-aware vs base, also paired. Both use a cell-line bootstrap stratified by sex.

The graph-fusion notebook uses the same pipeline, cohort, seeds and file formats, so the two can be merged into one fusion benchmark. Predictions are in harness format and are scored by `harness_results/evaluate.py` at the end. Use a GPU runtime; finished runs are skipped on re-run.

## 1. Setup

In [ ]:
# --- Setup: Colab mounts Drive, installs RDKit, clones the team repo. Locally: run from the repo root. ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/aljeralmazan28-web/Algoverse-Research-TNAM'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'], check=True)
    REPO_DIR = Path('/content/Algoverse-Research-TNAM')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = (Path('.') if Path('gdsc2_frozen_splits').exists() else Path('Algoverse-Research-TNAM')).resolve()
print('repo:', REPO_DIR)

### Inputs and sex labels
**Inputs.** Everything except the omics matrices comes from the repo: frozen GDSC2 splits (`gdsc2_frozen_splits/splits`), sex labels, the shared `sex_aware/` modules, the Week 2 `MultiOmicsPreprocessor`, DeepCDR's drug SMILES, and the evaluation harness. The four `*_aligned.csv` matrices (rows = SIDM IDs) come from the Week 2 package on Drive (`week2_aligned_omics_package.zip`), so point `OMICS_DIR` / `OMICS_ZIP` at it.

**`SEX_LABELS`** sets the label used for the sex-aware inputs, the reweighting, *and* per-sex evaluation:
* `'sensitivity'` (default): `harness_results/corrected_sex_labels_sensitivity.csv` (216 F / 253 M / 23 Unknown)
* `'strict'`: `harness_results/corrected_sex_labels.csv`; uncertain lines become Unknown (99 F / 137 M / 256 Unknown)
* `'reported'`: Cellosaurus reported donor sex shipped in the split files

Unknown lines stay in training (sex input = "unknown" indicator, weight 1) and are left out of the per-sex metrics. **To discuss:** which file is the primary analysis.

In [ ]:
# ======================= EDIT THESE =======================
DRIVE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.')
OMICS_DIR = DRIVE / 'week3_data'                        # holds expression/mutation/cnv/proteomics_aligned.csv (searched recursively)
OMICS_ZIP = DRIVE / 'week2_aligned_omics_package.zip'   # extracted into OMICS_DIR if the CSVs aren't there yet
CACHE_DIR = DRIVE / 'tnam_cache'                        # shared by both notebooks (drug SMILES cache)
OUT_DIR   = DRIVE / 'tnam_late_fusion'
SMILES_OVERRIDES = CACHE_DIR / 'drug_smiles_overrides.csv'  # optional: columns drug_id,smiles for drugs PubChem can't resolve
XY_CSV = None   # optional: output of sex_aware/export_xy.py (ModelID x X/Y genes, real DepMap expression)

SPLITS_TO_RUN = ['unseen_cell_lines']    # main result. Add 'unseen_tissues', 'unseen_drugs' when ready
SEEDS = [42, 123, 456]                   # same seeds as the transformer fusion runs
SEX_LABELS = 'sensitivity'               # 'sensitivity' | 'strict' | 'reported'  (see the note above)
RESTRICT_TO_PROTEOMICS_CELLS = True      # same cell cohort in both notebooks, so GF and LF results are comparable
PROT_MIN_CELL_COVERAGE = 0.5             # a cell "has proteomics" if >= 50% of proteins are measured
PROT_FEATURE_MIN_COVERAGE = 0.70         # Week 2 pipeline rule for keeping a protein
N_FEATURES = dict(expression=2000, cnv=2000, proteomics=2000, mutation=1000)   # top-variance, chosen on TRAIN cells
MIN_MUT_LINES = 5                        # keep a mutation feature only if >= 5 training lines carry it
MAX_ATOMS_CAP = 150                      # drugs with more heavy atoms are dropped (DeepCDR used 100)

TRAIN = dict(epochs=40, patience=6, batch_size=512, lr=1e-3, weight_decay=1e-4, grad_clip=5.0,
             max_train_rows=None)        # e.g. 20000 for a quick look; None = all training rows
RUN_COMPONENT_ABLATIONS = False          # True: also run each sex-aware component alone + the shuffled-sex control
SAVE_CHECKPOINTS = False
RUN_HARNESS = True                       # score predictions with harness_results/evaluate.py at the end
N_BOOT = 1000
# ---- late fusion ----
FAMILIES = ['DeepCDR', 'MOLI']
LF_LAYER_SETS = {'noprot': ['expression', 'mutation', 'cnv'],
                 'prot':   ['expression', 'mutation', 'cnv', 'proteomics']}
SMOKE_LAYERS = LF_LAYER_SETS['prot']
assert RESTRICT_TO_PROTEOMICS_CELLS, 'H3 needs both arms on the same proteomics-measured cell lines'

In [ ]:
import json, time, copy, random, re, hashlib, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
warnings.filterwarnings('ignore', category=FutureWarning)

# Team modules, imported straight from the repo so every fusion model shares one implementation
sys.path.insert(0, str(REPO_DIR / 'MultiOmics'))
sys.path.insert(0, str(REPO_DIR / 'sex_aware'))
from sex_aware import encode_sex, build_sex_aware, concat_sex_aware, sex_aware_extra_dim, weighted_mse
from reweighting import fit_sex_tissue_weights, apply_weights
from sex_shuffle import shuffle_sex_labels
from multiomics_preprocessing_pipeline import MultiOmicsPreprocessor

BASE_SEX_CFG = yaml.safe_load(open(REPO_DIR / 'sex_aware' / 'sex_aware_config.yaml'))['sex_aware']
assert BASE_SEX_CFG.get('sex_label_mode', 'unknown_indicator') == 'unknown_indicator', \
    "'exclude' mode would change the test cohort between runs; keep unknown_indicator"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for p in (CACHE_DIR, OUT_DIR, OUT_DIR / 'predictions', OUT_DIR / 'checkpoints'):
    p.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT_DIR / 'results.csv'
SPLIT_ROOT = REPO_DIR / 'gdsc2_frozen_splits' / 'splits'
SEXES = ('Female', 'Male')
print('device:', DEVICE, '| outputs ->', OUT_DIR)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def norm_sex(v):
    return {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}.get(str(v).strip().lower(), 'Unknown')

## 2. Frozen splits, sex labels, omics

In [ ]:
def load_split(name):
    return {p: pd.read_csv(SPLIT_ROOT / name / f'{p}.csv',
                           dtype={'sample_id': str, 'cell_line_id': str, 'drug_id': str})
            for p in ('train', 'val', 'test')}

# All three split designs partition the same 125,360 GDSC2 observations, so one design gives the full cohort.
ALL_ROWS = pd.concat(load_split('unseen_cell_lines').values(), ignore_index=True)
print(f"{len(ALL_ROWS):,} observations | {ALL_ROWS.cell_line_id.nunique()} cell lines | "
      f"{ALL_ROWS.drug_id.nunique()} drugs | {ALL_ROWS.tissue.nunique()} tissues")

SEX_LABEL_FILE = {'sensitivity': REPO_DIR / 'harness_results' / 'corrected_sex_labels_sensitivity.csv',
                  'strict':      REPO_DIR / 'harness_results' / 'corrected_sex_labels.csv',
                  'reported':    None}[SEX_LABELS]

META = (ALL_ROWS.drop_duplicates('cell_line_id').set_index('cell_line_id')[['tissue', 'sex']]
        .rename(columns={'sex': 'sex_reported'}))
META['sex_reported'] = META.sex_reported.map(norm_sex)
if SEX_LABEL_FILE is None:
    META['sex'] = META.sex_reported
else:
    lab = pd.read_csv(SEX_LABEL_FILE, dtype={'cell_line_id': str})
    assert not lab.cell_line_id.duplicated().any()
    META['sex'] = lab.set_index('cell_line_id')['corrected_sex'].map(norm_sex).reindex(META.index).fillna('Unknown')
print('sex label source:', SEX_LABELS, '->', SEX_LABEL_FILE)
pd.crosstab(META.sex_reported, META.sex, margins=True, rownames=['reported'], colnames=['used'])

In [ ]:
OMICS_FILES = dict(expression='expression_aligned.csv', mutation='mutation_aligned.csv',
                   cnv='cnv_aligned.csv', proteomics='proteomics_aligned.csv')

def find_omics_file(fname):
    hits = sorted(OMICS_DIR.rglob(fname)) if OMICS_DIR.exists() else []
    return hits[0] if hits else None

if any(find_omics_file(f) is None for f in OMICS_FILES.values()) and OMICS_ZIP.exists():
    OMICS_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(OMICS_ZIP) as z:
        z.extractall(OMICS_DIR)
missing = [f for f in OMICS_FILES.values() if find_omics_file(f) is None]
assert not missing, f'Missing omics files {missing} under {OMICS_DIR}. Edit OMICS_DIR / OMICS_ZIP in the config cell.'

def read_omics(path):
    df = pd.read_csv(path, index_col=0, low_memory=False)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    df = df.loc[~df.index.duplicated(), ~df.columns.duplicated()]
    try:
        return df.astype('float32')
    except (ValueError, TypeError):
        return df.apply(pd.to_numeric, errors='coerce').astype('float32')

OMICS = {k: read_omics(find_omics_file(f)) for k, f in OMICS_FILES.items()}
OMICS['mutation'] = (OMICS['mutation'].fillna(0) != 0).astype('float32')     # binary mutation status
for k, v in OMICS.items():
    print(f'{k:11s} {v.shape}  NaN fraction {float(v.isna().mean().mean()):.3f}')

# Cohort: cell lines in the frozen splits that have every omics layer
cohort = set(META.index)
for v in OMICS.values():
    cohort &= set(v.index)
print('split cell lines with all four omics matrices:', len(cohort), 'of', len(META))
if RESTRICT_TO_PROTEOMICS_CELLS:
    prot_cov = OMICS['proteomics'].notna().mean(axis=1)
    cohort &= set(prot_cov.index[prot_cov >= PROT_MIN_CELL_COVERAGE])
    print(f'... with >= {PROT_MIN_CELL_COVERAGE:.0%} proteins measured:', len(cohort))
COHORT_IDS = sorted(cohort)
CELL_POS = {c: i for i, c in enumerate(COHORT_IDS)}
print(META.loc[COHORT_IDS].groupby('sex').size().to_dict())

## Drug graphs
DeepCDR's shipped `drug_graph_feat` covers only 70 of the 295 GDSC2 drugs (about 23% of observations), so graphs are rebuilt from SMILES for every drug with one featurizer, so all drugs are encoded the same way:

1. `drug_smiles_overrides.csv` (manual fixes, optional) → 2. cached lookups → 3. DeepCDR's `223drugs_pubchem_smiles.txt` via the GDSC drug list → 4. PubChem name lookup (needs internet, so Colab).

Atoms get DeepCDR/deepchem's 75 features (element, degree, implicit valence, charge, radicals, hybridisation, aromaticity, H count); the adjacency is the GCN-normalised `D^-1/2 (A+I) D^-1/2`. Salts are reduced to the largest fragment. Drugs that still have no SMILES are dropped and the coverage table shows how many rows that costs.

In [ ]:
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
import requests, urllib.parse

DRUG_NAMES = (pd.read_csv(REPO_DIR / 'gdsc2_frozen_splits' / 'source_data' / 'gdsc_response_mapping_audit.csv',
                          usecols=['drug_id', 'drug_name'], dtype={'drug_id': str})
              .dropna().drop_duplicates('drug_id').set_index('drug_id')['drug_name'])
NEEDED = sorted(ALL_ROWS.drug_id.unique(), key=int)
SMILES, SOURCE = {}, {}

def _take(d, smi, src):
    if d not in SMILES and isinstance(smi, str) and smi.strip():
        SMILES[d], SOURCE[d] = smi.strip(), src

if SMILES_OVERRIDES.exists():
    for r in pd.read_csv(SMILES_OVERRIDES, dtype=str).itertuples():
        _take(r.drug_id, r.smiles, 'override')
CACHE_FILE = CACHE_DIR / 'drug_smiles.csv'
if CACHE_FILE.exists():
    for r in pd.read_csv(CACHE_FILE, dtype=str).itertuples():
        _take(r.drug_id, r.smiles, r.source)

dl = pd.read_csv(REPO_DIR / 'GDSC' / '1.Drug_listMon Jun 24 09_00_55 2019.csv', encoding='latin1', dtype=str)
pc = pd.read_csv(REPO_DIR / 'DeepCDR' / 'data' / '223drugs_pubchem_smiles.txt', sep='\t', header=None,
                 names=['pubchem', 'smiles'], dtype=str)
pc_smiles = dict(zip(pc.pubchem.str.strip(), pc.smiles))
for r in dl.itertuples():
    _take(str(r.drug_id).strip(), pc_smiles.get(str(r.PubCHEM).strip()), 'deepcdr_pubchem')

def pubchem_smiles(name):
    variants = [name, re.sub(r'\s*\(.*?\)\s*', ' ', name).strip()]
    for v in dict.fromkeys(x for x in variants if x):
        q = urllib.parse.quote(v, safe='')
        for prop in ('SMILES', 'IsomericSMILES', 'CanonicalSMILES'):   # property names changed across PubChem versions
            try:
                r = requests.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{q}/property/{prop}/TXT',
                                 timeout=20)
            except requests.RequestException:
                return None
            time.sleep(0.25)                                            # PubChem asks for <= 5 requests/second
            if r.status_code == 200 and r.text.strip():
                return r.text.strip().splitlines()[0].strip()
            if r.status_code == 404:                                    # name not found: try next variant
                break
    return None

todo = [d for d in NEEDED if d not in SMILES and d in DRUG_NAMES.index]
print(f'{len(NEEDED) - len(todo)} drugs resolved locally; looking up {len(todo)} on PubChem...')
for d in todo:
    _take(d, pubchem_smiles(DRUG_NAMES[d]), 'pubchem_name')
pd.DataFrame([dict(drug_id=d, drug_name=DRUG_NAMES.get(d, ''), smiles=SMILES[d], source=SOURCE[d])
              for d in SMILES if SOURCE[d] != 'override']).to_csv(CACHE_FILE, index=False)

# ---- featurize (deepchem / DeepCDR 75-d atom features) ----
SYMBOLS = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe', 'As', 'Al', 'I', 'B', 'V', 'K',
           'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti', 'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd',
           'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb', 'Unknown']
HYB = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP3,
       Chem.rdchem.HybridizationType.SP3D, Chem.rdchem.HybridizationType.SP3D2]

def _onehot(x, allowed, unk=False):
    v = [float(x == a) for a in allowed]
    if unk and not any(v):
        v[-1] = 1.0
    return v

def _implicit_valence(a):
    try:
        return a.GetValence(Chem.ValenceType.IMPLICIT)
    except AttributeError:
        return a.GetImplicitValence()

def atom_features(a):
    f = (_onehot(a.GetSymbol(), SYMBOLS, unk=True) + _onehot(a.GetDegree(), list(range(11)))
         + _onehot(_implicit_valence(a), list(range(7))) + [float(a.GetFormalCharge()), float(a.GetNumRadicalElectrons())]
         + _onehot(a.GetHybridization(), HYB) + [float(a.GetIsAromatic())] + _onehot(a.GetTotalNumHs(), list(range(5)), unk=True))
    assert len(f) == 75
    return f

def mol_graph(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True)
    mol = max(frags, key=lambda m: m.GetNumAtoms())
    n = mol.GetNumAtoms()
    if n == 0 or n > MAX_ATOMS_CAP:
        return None
    x = np.array([atom_features(a) for a in mol.GetAtoms()], dtype=np.float32)
    A = np.eye(n, dtype=np.float32)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        A[i, j] = A[j, i] = 1.0
    dinv = 1.0 / np.sqrt(A.sum(1))
    return x, A * dinv[:, None] * dinv[None, :]

graphs = {d: g for d in NEEDED if d in SMILES and (g := mol_graph(SMILES[d])) is not None}
DRUG_IDS = sorted(graphs, key=int)
DRUG_POS = {d: i for i, d in enumerate(DRUG_IDS)}
MAX_ATOMS = max(graphs[d][0].shape[0] for d in DRUG_IDS)
DRUG_X = np.zeros((len(DRUG_IDS), MAX_ATOMS, 75), np.float32)
DRUG_ADJ = np.zeros((len(DRUG_IDS), MAX_ATOMS, MAX_ATOMS), np.float32)
DRUG_MASK = np.zeros((len(DRUG_IDS), MAX_ATOMS), np.float32)
for i, d in enumerate(DRUG_IDS):
    x, a = graphs[d]; n = len(x)
    DRUG_X[i, :n], DRUG_ADJ[i, :n, :n], DRUG_MASK[i, :n] = x, a, 1.0

DRUG_TABLE = pd.DataFrame([dict(drug_id=d, drug_name=DRUG_NAMES.get(d, ''), smiles=SMILES.get(d, ''),
                                source=SOURCE.get(d, 'unresolved'), has_graph=d in DRUG_POS) for d in NEEDED])
DRUG_TABLE.to_csv(OUT_DIR / 'drug_table.csv', index=False)
print(f'drugs with a graph: {len(DRUG_IDS)}/{len(NEEDED)} | max atoms {MAX_ATOMS} | '
      f'observations covered: {ALL_ROWS.drug_id.isin(DRUG_POS).mean():.1%}')
print(DRUG_TABLE.source.value_counts().to_dict())
print('unresolved (add to drug_smiles_overrides.csv):', DRUG_TABLE.loc[~DRUG_TABLE.has_graph, 'drug_name'].tolist())

## X/Y features

In [ ]:
# X/Y features for the XY module (sex_aware.XYModule). Priority: XY_CSV from export_xy.py ->
# X/Y genes found in the expression matrix -> Week 2 xist_score / y_score from sex_labels_final.csv.
XY_GENE_LIST = ['XIST', 'RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY', 'EIF1AY', 'ZFY', 'USP9Y', 'TXLNGY', 'NLGN4Y']
Y_GENES = set(XY_GENE_LIST) - {'XIST'}

# SIDM <-> DepMap ModelID via Cellosaurus accession (both files are in the repo)
_aud = pd.read_csv(REPO_DIR / 'gdsc2_frozen_splits' / 'source_data' / 'cellosaurus_sex_mapping_audit.csv', dtype=str)
_fin = pd.read_csv(REPO_DIR / 'sex_labels' / 'data' / 'sex_labels_final.csv', dtype={'ModelID': str})
SIDM_MAP = (_aud[['cell_line_id', 'cellosaurus_id']].dropna().drop_duplicates()
            .merge(_fin, left_on='cellosaurus_id', right_on='cvcl_final', how='inner')
            .drop_duplicates('cell_line_id').set_index('cell_line_id'))

def build_xy_table():
    if XY_CSV and Path(XY_CSV).exists():
        t = pd.read_csv(XY_CSV, index_col=0)
        m = SIDM_MAP.reset_index().set_index('ModelID')['cell_line_id']
        t.index = t.index.map(m.to_dict())
        return t[t.index.notna()], f'XY_CSV ({Path(XY_CSV).name})'
    expr = OMICS['expression']
    clean = [c.split(' (')[0] for c in expr.columns]
    pos = {}
    for i, g in enumerate(clean):
        pos.setdefault(g, i)
    found = [g for g in XY_GENE_LIST if g in pos]
    print('X/Y genes found in expression matrix:', found, '| missing:', [g for g in XY_GENE_LIST if g not in pos])
    if 'XIST' in found and Y_GENES & set(found):
        t = expr.iloc[:, [pos[g] for g in found]].copy(); t.columns = found
        return t, 'expression_aligned.csv columns'
    return SIDM_MAP[['xist_score', 'y_score']].astype(float), 'sex_labels_final.csv xist_score/y_score'

XY_TABLE, XY_SOURCE = build_xy_table()
XY_TABLE = XY_TABLE.reindex(COHORT_IDS)
print('XY source:', XY_SOURCE, '| features:', list(XY_TABLE.columns),
      '| cohort lines with XY data:', int(XY_TABLE.notna().any(axis=1).sum()), '/', len(COHORT_IDS))
# Sanity check: XIST-like feature should be higher in Female lines, Y genes higher in Male lines
print(XY_TABLE.join(META['sex']).groupby('sex').mean().round(3))

## Per-split preprocessing (fit on training cell lines only)
For each split design: drop rows whose cell line is outside the cohort or whose drug has no graph → choose features on **training cells only** (top variance; proteins need ≥70% coverage; mutations need ≥5 mutated lines) → Week 2 `MultiOmicsPreprocessor` (median impute, drop constants, z-score expression/proteomics) → z-score CNV with training statistics → clip to ±10. The target (`response`, ln IC50) is standardised with training mean/SD for optimisation and converted back for every metric.

In [ ]:
def prepare_split(split_name):
    parts = load_split(split_name)
    rows, coverage = {}, []
    for p, df in parts.items():
        ok = df.cell_line_id.isin(CELL_POS) & df.drug_id.isin(DRUG_POS)
        coverage.append(dict(split=split_name, partition=p, rows=len(df), rows_kept=int(ok.sum()),
                             frac_kept=round(float(ok.mean()), 4), cells=df.cell_line_id.nunique(),
                             cells_kept=df.loc[ok, 'cell_line_id'].nunique(), drugs=df.drug_id.nunique(),
                             drugs_kept=df.loc[ok, 'drug_id'].nunique()))
        df = df[ok]
        rows[p] = dict(sample_id=df.sample_id.to_numpy(), cell_line_id=df.cell_line_id.to_numpy(),
                       drug_id=df.drug_id.to_numpy(), tissue=df.tissue.to_numpy(),
                       cell=df.cell_line_id.map(CELL_POS).to_numpy(np.int64),
                       drug=df.drug_id.map(DRUG_POS).to_numpy(np.int64),
                       y=df.response.to_numpy(np.float64))
        rows[p]['sex'] = META.loc[rows[p]['cell_line_id'], 'sex'].to_numpy()      # evaluation label
    train_ids = sorted(set(rows['train']['cell_line_id']))
    train_pos = np.array([CELL_POS[c] for c in train_ids])

    # feature selection on training cells only
    sel = {}
    for k, df in OMICS.items():
        tr = df.loc[train_ids]
        if k == 'mutation':
            cnt = (tr != 0).sum()
            sel[k] = list(cnt[cnt >= MIN_MUT_LINES].sort_values(ascending=False).index[:N_FEATURES[k]])
        else:
            if k == 'proteomics':
                tr = tr.loc[:, tr.notna().mean() >= PROT_FEATURE_MIN_COVERAGE]
            v = tr.var(skipna=True).dropna()
            sel[k] = list(v[v > 0].sort_values(ascending=False).index[:N_FEATURES[k]])
    sub = {k: OMICS[k][sel[k]] for k in OMICS}
    proc = MultiOmicsPreprocessor(protein_min_coverage=PROT_FEATURE_MIN_COVERAGE).fit(sub, train_ids)
    out = proc.transform(sub, COHORT_IDS)
    X = {}
    for k, df in out.items():
        a = df.to_numpy(np.float64)
        if k == 'cnv':                                          # pipeline leaves CNV unscaled
            a = (a - a[train_pos].mean(0)) / (a[train_pos].std(0) + 1e-6)
        X[k] = np.clip(np.nan_to_num(a), -10, 10).astype(np.float32)

    xy = XY_TABLE.to_numpy(np.float64)
    mu, sd = np.nanmean(xy[train_pos], 0), np.nanstd(xy[train_pos], 0) + 1e-6
    xy = np.clip(np.nan_to_num((xy - mu) / sd), -10, 10).astype(np.float32)   # missing -> training mean

    tr = rows['train']
    med = pd.Series(tr['y']).groupby(tr['drug']).median()
    tr['sensitive'] = (tr['y'] < med.reindex(tr['drug']).to_numpy()).astype(np.int64)   # MOLI triplet label

    data = dict(split=split_name, rows=rows, X=X, dims={k: v.shape[1] for k, v in X.items()},
                features={k: list(df.columns) for k, df in out.items()}, xy=xy,
                y_mu=float(tr['y'].mean()), y_sd=float(tr['y'].std()),
                cell_sex=META.loc[COHORT_IDS, 'sex'].to_numpy(object),
                cell_tissue=META.loc[COHORT_IDS, 'tissue'].to_numpy(object),
                coverage=pd.DataFrame(coverage),
                train_sha256=sha256(SPLIT_ROOT / split_name / 'train.csv'),
                val_sha256=sha256(SPLIT_ROOT / split_name / 'val.csv'))
    data['coverage'].to_csv(OUT_DIR / f'coverage_{split_name}.csv', index=False)
    return data

def to_tensors(data):
    t = lambda a: torch.as_tensor(a, device=DEVICE)
    return dict(omics={k: t(v) for k, v in data['X'].items()}, xy=t(data['xy']),
                drug_x=t(DRUG_X), drug_adj=t(DRUG_ADJ), drug_mask=t(DRUG_MASK))

## Shared building blocks
* **DrugGCN** – DeepCDR's drug branch: stacked graph convolutions on the molecular graph, masked global max-pool. Used by both families (MOLI was drug-specific and had no drug encoder; a pan-drug model needs one, and sharing it keeps the comparison about omics encoding and fusion).
* **DeepCDR omics branches** – expression and CNV/proteomics: `Dense(256, tanh) → BN → Dropout(0.1) → Dense(100, relu)`; mutation: DeepCDR's 1-D CNN (kernel and stride scaled down because our mutation input is gene-level, ~1k features, not DeepCDR's 34k loci). CNV stands in for DeepCDR's methylation branch.
* **DeepCDR head** – `Dense(300)` then three 1-D conv/pool blocks, as in the paper.
* **MOLI encoders** – one `Linear → BN → ReLU → Dropout` per omic (MOLI's design), L2-normalised fused embedding, and a **triplet loss** adapted to regression: within a batch, a pair is "sensitive" if its ln IC50 is below that drug's training median.
* **MOLI head** – MOLI used one linear layer because each model covered one drug. A linear layer on `[cell, drug]` cannot represent cell×drug interactions, so the pan-drug version uses a one-hidden-layer MLP.

In [ ]:
class DrugGCN(nn.Module):
    def __init__(self, in_dim=75, hidden=(256, 256, 256), out_dim=100, dropout=0.1):
        super().__init__()
        dims = [in_dim, *hidden]
        self.lins = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.norms = nn.ModuleList(nn.LayerNorm(b) for b in hidden)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden[-1], out_dim)

    def forward(self, x, adj, mask):                       # x [B,A,75], adj [B,A,A], mask [B,A]
        m = mask.unsqueeze(-1)
        for lin, ln in zip(self.lins, self.norms):
            x = self.drop(F.relu(ln(torch.bmm(adj, lin(x))))) * m
        x = x.masked_fill(m == 0, float('-inf')).max(dim=1).values
        return F.relu(self.out(x))

def embed_drugs(gcn, batch):
    """Run the GCN once per distinct drug in the batch, then scatter back to rows."""
    uniq, inv = torch.unique(batch['drug_idx'], return_inverse=True)
    return gcn(batch['drug_x'][uniq], batch['drug_adj'][uniq], batch['drug_mask'][uniq])[inv]

class DenseBranch(nn.Module):                               # DeepCDR gene-expression / methylation subnetwork
    def __init__(self, d_in, hidden=256, out=100, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.Tanh(), nn.BatchNorm1d(hidden), nn.Dropout(dropout),
                                 nn.Linear(hidden, out), nn.ReLU())
    def forward(self, x):
        return self.net(x)

class MutConvBranch(nn.Module):                             # DeepCDR mutation subnetwork (1-D CNN)
    def __init__(self, d_in, out=100):
        super().__init__()
        assert d_in >= 8, 'too few mutation features for the conv branch'
        k1 = min(50, max(3, d_in // 20)); s1 = max(1, min(5, d_in // 200))
        self.conv = nn.Sequential(nn.Conv1d(1, 50, k1, stride=s1), nn.Tanh(), nn.MaxPool1d(s1),
                                  nn.Conv1d(50, 30, 5, stride=2), nn.ReLU(), nn.AdaptiveMaxPool1d(8), nn.Flatten())
        self.fc = nn.Sequential(nn.Linear(30 * 8, out), nn.ReLU())
    def forward(self, x):
        return self.fc(self.conv(x.unsqueeze(1)))

class DeepCDRHead(nn.Module):
    def __init__(self, d_in, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d_in, 300), nn.Tanh(), nn.Dropout(dropout))
        self.conv = nn.Sequential(nn.Conv1d(1, 30, 150), nn.ReLU(), nn.MaxPool1d(2),
                                  nn.Conv1d(30, 10, 5), nn.ReLU(), nn.MaxPool1d(3),
                                  nn.Conv1d(10, 5, 5), nn.ReLU(), nn.MaxPool1d(3),
                                  nn.Dropout(dropout), nn.Flatten(), nn.Dropout(0.2))
        self.out = nn.Linear(5 * 6, 1)                      # 300 -> 151 -> 75 -> 71 -> 23 -> 19 -> 6
    def forward(self, z):
        return self.out(self.conv(self.fc(z).unsqueeze(1))).squeeze(-1)

class MOLIEncoder(nn.Module):
    def __init__(self, d_in, h, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class MLPHead(nn.Module):
    def __init__(self, d_in, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, z):
        return self.net(z).squeeze(-1)

DEEPCDR_HP = dict(drug_hidden=(256, 256, 256), drug_dim=100, branch_dim=100, dropout=0.1)
MOLI_HP = dict(h=dict(expression=256, mutation=128, cnv=128, proteomics=256), dropout=0.4,
               drug_hidden=(128, 128, 128), drug_dim=128, triplet_weight=0.2, triplet_margin=0.5)

def build_encoders(family, layers, dims):
    """Returns (ModuleDict of omics encoders, {layer: output width}, drug GCN, drug output width)."""
    if family == 'DeepCDR':
        hp = DEEPCDR_HP
        enc = {k: MutConvBranch(dims[k], hp['branch_dim']) if k == 'mutation'
               else DenseBranch(dims[k], 256, hp['branch_dim'], hp['dropout']) for k in layers}
        out = {k: hp['branch_dim'] for k in layers}
    elif family == 'MOLI':
        hp = MOLI_HP
        enc = {k: MOLIEncoder(dims[k], hp['h'][k], hp['dropout']) for k in layers}
        out = {k: hp['h'][k] for k in layers}
    else:
        raise ValueError(family)
    return nn.ModuleDict(enc), out, DrugGCN(hidden=hp['drug_hidden'], out_dim=hp['drug_dim']), hp['drug_dim']

def batch_hard_triplet(z, labels, margin):
    """MOLI-style triplet loss with batch-hard mining on L2-normalised embeddings."""
    z = F.normalize(z, dim=1)
    d = torch.sqrt((2 - 2 * z @ z.T).clamp_min(1e-12))
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    pos, neg = same & ~eye, ~same
    valid = pos.any(1) & neg.any(1)
    if not valid.any():
        return z.new_zeros(())
    hardest_pos = d.masked_fill(~pos, float('-inf')).max(1).values
    hardest_neg = d.masked_fill(~neg, float('inf')).min(1).values
    return F.relu(hardest_pos - hardest_neg + margin)[valid].mean()

## Late-fusion model
`LF_DeepCDR` is DeepCDR's own architecture (drug GCN + per-omic subnetworks concatenated into the conv head), with CNV in place of methylation and an optional proteomics branch. `LF_MOLI` follows MOLI: separate encoders, concatenation, L2 normalisation. The sex-aware add-ons attach after concatenation through `concat_sex_aware`, which is where `sex_aware.py` says late fusion should take them.

In [ ]:
class LateFusionDRP(nn.Module):
    def __init__(self, family, layers, dims, n_xy, sex_cfg):
        super().__init__()
        self.family, self.layers = family, list(layers)
        self.encoders, enc_out, self.drug, drug_out = build_encoders(family, self.layers, dims)
        self.sex_input, self.xy_module = build_sex_aware(sex_cfg, n_xy)
        d_fused = sum(enc_out.values()) + drug_out + sex_aware_extra_dim(self.sex_input, self.xy_module)
        self.head = DeepCDRHead(d_fused) if family == 'DeepCDR' else MLPHead(d_fused)

    def forward(self, batch):
        parts = [self.encoders[k](batch['omics'][k]) for k in self.layers]
        drug = embed_drugs(self.drug, batch)
        if self.family == 'MOLI':
            z = torch.cat([F.normalize(torch.cat(parts, 1), dim=1), F.normalize(drug, dim=1)], 1)
        else:
            z = torch.cat(parts + [drug], 1)
        emb = z
        z = concat_sex_aware(z, self.sex_input, self.xy_module, batch['sex'], batch['xy'])
        return dict(pred=self.head(z), emb=emb)

def build_model(family, layers, data, sex_cfg):
    return LateFusionDRP(family, layers, data['dims'], data['xy'].shape[1], sex_cfg)

## Training, metrics, run logging

In [ ]:
def make_sex_inputs(data, sex_cfg, seed):
    """Model-visible sex labels (shuffled within tissue for the control run) -> one-hot [F, M, unknown]."""
    labels = data['cell_sex'].copy()
    if sex_cfg.get('shuffle_sex_labels'):
        labels = shuffle_sex_labels(labels, data['cell_tissue'],
                                    within_tissue=sex_cfg.get('shuffle_within_tissue', True), seed=seed)
    feats, _ = encode_sex(list(labels), mode='unknown_indicator')
    return labels, feats.to(DEVICE)

def make_row_weights(data, model_labels):
    """Team reweighting: sex x tissue weights fitted on TRAINING cell lines, repeated per (line, drug) row."""
    tr = data['rows']['train']
    cells = np.unique(tr['cell'])
    lines = pd.DataFrame({'tissue': data['cell_tissue'][cells], 'sex': model_labels[cells]})
    table = fit_sex_tissue_weights(lines, sex_col='sex', tissue_col='tissue')
    rows = pd.DataFrame({'tissue': data['cell_tissue'][tr['cell']], 'sex': model_labels[tr['cell']]})
    return apply_weights(rows, table, sex_col='sex', tissue_col='tissue').to_numpy(np.float32)

def make_batch(T, layers, c, d, sex_t):
    return dict(omics={k: T['omics'][k][c] for k in layers}, drug_idx=d, drug_x=T['drug_x'],
                drug_adj=T['drug_adj'], drug_mask=T['drug_mask'], sex=sex_t[c], xy=T['xy'][c])

def rmse(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.sqrt(np.mean((y - p) ** 2))) if len(y) else np.nan

@torch.no_grad()
def predict(model, T, part, sex_t, data, bs=4096):
    model.eval()
    preds, attn = [], []
    for i in range(0, len(part['y']), bs):
        c = torch.as_tensor(part['cell'][i:i + bs], device=DEVICE)
        d = torch.as_tensor(part['drug'][i:i + bs], device=DEVICE)
        out = model(make_batch(T, model.layers, c, d, sex_t))
        preds.append(out['pred'].float().cpu().numpy())
        if out.get('node_attn') is not None:
            attn.append(out['node_attn'].float().cpu().numpy())
    pred = np.concatenate(preds) * data['y_sd'] + data['y_mu']
    return dict(pred=pred, attn=np.concatenate(attn) if attn else None)

def train_model(model, T, data, sex_t, weights, triplet, cfg=None, verbose=True):
    cfg = cfg or TRAIN
    tr, va = data['rows']['train'], data['rows']['val']
    idx = np.arange(len(tr['y']))
    if cfg['max_train_rows'] and len(idx) > cfg['max_train_rows']:
        idx = np.sort(np.random.default_rng(0).choice(idx, cfg['max_train_rows'], replace=False))
    dev = lambda a, dt=None: torch.as_tensor(a, device=DEVICE, dtype=dt)
    c, d = dev(tr['cell'][idx]), dev(tr['drug'][idx])
    y = dev((tr['y'][idx] - data['y_mu']) / data['y_sd'], torch.float32)
    w = None if weights is None else dev(weights[idx], torch.float32)
    sens = dev(tr['sensitive'][idx])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    best_rmse, best_state, best_epoch, bad, hist = np.inf, None, -1, 0, []
    for ep in range(cfg['epochs']):
        model.train()
        perm = torch.randperm(len(y), device=DEVICE)
        tot, nb, t0 = 0.0, 0, time.time()
        for i in range(0, len(perm), cfg['batch_size']):
            b = perm[i:i + cfg['batch_size']]
            if len(b) < 2:                                   # BatchNorm needs > 1 row
                continue
            out = model(make_batch(T, model.layers, c[b], d[b], sex_t))
            loss = weighted_mse(out['pred'], y[b], None if w is None else w[b])
            if triplet:
                loss = loss + triplet['weight'] * batch_hard_triplet(out['emb'], sens[b], triplet['margin'])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step()
            tot += loss.item(); nb += 1
        v = rmse(va['y'], predict(model, T, va, sex_t, data)['pred'])
        hist.append(dict(epoch=ep, train_loss=tot / max(nb, 1), val_rmse=v))
        if verbose:
            print(f'  epoch {ep:2d}  loss {tot / max(nb, 1):.4f}  val RMSE {v:.4f}  ({time.time() - t0:.0f}s)')
        if v < best_rmse - 1e-4:
            best_rmse, best_epoch, bad = v, ep, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg['patience']:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return dict(best_val_rmse=best_rmse, best_epoch=best_epoch, epochs_run=len(hist), history=hist)

In [ ]:
def corr(y, p):
    if len(y) < 3 or np.std(y) == 0 or np.std(p) == 0:
        return np.nan, np.nan
    return float(pearsonr(y, p)[0]), float(spearmanr(y, p)[0])

def per_cell(df):
    return (df.assign(se=(df.y_true - df.y_pred) ** 2).groupby('cell_line_id')
              .agg(sse=('se', 'sum'), n=('se', 'size'), sex=('sex', 'first'), tissue=('tissue', 'first')))

def _boot_counts(g, groups, n_boot, rng):
    """Cell-line bootstrap, resampling lines separately within each sex group (incl. Unknown)."""
    sse, n = {}, {}
    for s in groups:
        gs = g[g.sex == s]
        if len(gs) == 0:
            continue
        cnt = rng.multinomial(len(gs), np.full(len(gs), 1 / len(gs)), size=n_boot)
        sse[s], n[s] = cnt @ gs.sse.to_numpy(), cnt @ gs.n.to_numpy()
    return sse, n

def sex_metrics(df, n_boot=N_BOOT, seed=0):
    """Overall / Female / Male RMSE + correlations, signed gap (F - M) with a cell-line bootstrap CI."""
    out = {}
    for name, sub in [('all', df)] + [(s, df[df.sex == s]) for s in SEXES]:
        p, s = corr(sub.y_true.to_numpy(), sub.y_pred.to_numpy())
        out.update({f'{name}_rmse': rmse(sub.y_true, sub.y_pred), f'{name}_pearson': p, f'{name}_spearman': s,
                    f'{name}_n_obs': len(sub), f'{name}_n_lines': sub.cell_line_id.nunique()})
    out['gap_F_minus_M'] = out['Female_rmse'] - out['Male_rmse']
    out['gap_ci_low'] = out['gap_ci_high'] = np.nan
    sse, n = _boot_counts(per_cell(df), SEXES, n_boot, np.random.default_rng(seed))
    if len(sse) == 2:
        gaps = np.sqrt(sse['Female'] / n['Female']) - np.sqrt(sse['Male'] / n['Male'])
        out['gap_ci_low'], out['gap_ci_high'] = np.percentile(gaps, [2.5, 97.5])
    return out

def paired_compare(df_a, df_b, n_boot=N_BOOT, seed=0):
    """B minus A on identical test rows: delta RMSE overall / F / M, delta signed gap and delta |gap|, with a
    paired cell-line bootstrap (same resampled lines for both models). Negative delta RMSE = B better."""
    m = df_a[['sample_id', 'cell_line_id', 'sex', 'tissue', 'y_true', 'y_pred']].merge(
        df_b[['sample_id', 'y_pred']], on='sample_id', suffixes=('_a', '_b'), validate='one_to_one')
    assert len(m) == len(df_a) == len(df_b), 'models were not scored on identical test rows'
    rng_seed = np.random.default_rng(seed).integers(1 << 31)
    groups = sorted(m.sex.unique())
    res = {}
    stats = {}
    for tag in ('a', 'b'):
        g = per_cell(m.rename(columns={f'y_pred_{tag}': 'y_pred'}))
        sse, n = _boot_counts(g, groups, n_boot, np.random.default_rng(rng_seed))   # same seed -> same resamples
        point = {s: np.sqrt(g[g.sex == s].sse.sum() / g[g.sex == s].n.sum()) for s in SEXES}
        point['all'] = np.sqrt(g.sse.sum() / g.n.sum())
        boot = {s: np.sqrt(sse[s] / n[s]) for s in SEXES if s in sse}
        boot['all'] = np.sqrt(sum(sse.values()) / sum(n.values()))
        stats[tag] = (point, boot)
    (pa, ba), (pb, bb) = stats['a'], stats['b']
    def add(name, pt, bt):
        res[name] = pt
        res[name + '_ci_low'], res[name + '_ci_high'] = np.percentile(bt, [2.5, 97.5])
    for s in ('all',) + SEXES:
        add(f'delta_rmse_{s}', pb[s] - pa[s], bb[s] - ba[s])
    ga, gb = pa['Female'] - pa['Male'], pb['Female'] - pb['Male']
    gba, gbb = ba['Female'] - ba['Male'], bb['Female'] - bb['Male']
    add('delta_gap', gb - ga, gbb - gba)
    add('delta_abs_gap', abs(gb) - abs(ga), np.abs(gbb) - np.abs(gba))
    return res

def within_tissue(df, min_lines=3):
    g = per_cell(df)
    out = []
    for t, gt in g.groupby('tissue'):
        nf, nm = (gt.sex == 'Female').sum(), (gt.sex == 'Male').sum()
        if nf >= min_lines and nm >= min_lines:
            r = {s: np.sqrt(gt[gt.sex == s].sse.sum() / gt[gt.sex == s].n.sum()) for s in SEXES}
            out.append(dict(tissue=t, n_female_lines=nf, n_male_lines=nm, female_rmse=r['Female'],
                            male_rmse=r['Male'], gap_F_minus_M=r['Female'] - r['Male']))
    return pd.DataFrame(out)

def seed_mean_predictions(model_name, split):
    """Average test predictions over seeds (one ensemble per configuration)."""
    dfs = [pd.read_csv(f, dtype={'sample_id': str, 'cell_line_id': str, 'drug_id': str})
           for s in SEEDS if (f := OUT_DIR / 'predictions' / f'seed{s}' / f'{model_name}__{split}.csv').exists()]
    if not dfs:
        return None
    base = dfs[0].drop(columns='y_pred')
    preds = pd.concat([d.set_index('sample_id')['y_pred'] for d in dfs], axis=1).mean(axis=1)
    return base.assign(y_pred=base.sample_id.map(preds))

In [ ]:
def already_done(split, model_name, seed, pred_file):
    if not (pred_file.exists() and RESULTS_CSV.exists()):
        return False
    r = pd.read_csv(RESULTS_CSV)
    return bool(((r.split == split) & (r.model == model_name) & (r.seed == seed)).any())

def run_experiment(data, T, spec, seed, verbose=False):
    """spec: fusion, family, layers, layer_tag, sex_name, sex_cfg. Saves harness-format test predictions."""
    split = data['split']
    model_name = f"{spec['fusion']}_{spec['family']}_{spec['layer_tag']}_{spec['sex_name']}"
    pred_dir = OUT_DIR / 'predictions' / f'seed{seed}'; pred_dir.mkdir(parents=True, exist_ok=True)
    pred_file = pred_dir / f'{model_name}__{split}.csv'
    if already_done(split, model_name, seed, pred_file):
        print(f'skip (done): {model_name} {split} seed {seed}')
        return None
    set_seed(seed)
    sex_cfg = {**BASE_SEX_CFG, **spec['sex_cfg']}
    model_labels, sex_t = make_sex_inputs(data, sex_cfg, seed)
    weights = make_row_weights(data, model_labels) if sex_cfg.get('use_reweighting') else None
    model = build_model(spec['family'], spec['layers'], data, sex_cfg).to(DEVICE)
    triplet = (dict(weight=MOLI_HP['triplet_weight'], margin=MOLI_HP['triplet_margin'])
               if spec['family'] == 'MOLI' and MOLI_HP['triplet_weight'] > 0 else None)
    t0 = time.time()
    info = train_model(model, T, data, sex_t, weights, triplet, verbose=verbose)
    te = data['rows']['test']
    out = predict(model, T, te, sex_t, data)
    pred = pd.DataFrame(dict(sample_id=te['sample_id'], cell_line_id=te['cell_line_id'], drug_id=te['drug_id'],
                             tissue=te['tissue'], sex=te['sex'], y_true=te['y'], y_pred=out['pred']))
    if out['attn'] is not None:                              # graph fusion: drug-node attention per modality
        for j, node in enumerate(model.nodes):
            pred[f'attn_{node}'] = out['attn'][:, j]
    pred.to_csv(pred_file, index=False)
    if SAVE_CHECKPOINTS:
        torch.save(model.state_dict(), OUT_DIR / 'checkpoints' / f'{model_name}__{split}__s{seed}.pt')

    # provenance for the harness: these models really were trained on these frozen files
    man_file = pred_dir / 'model_runs.json'
    man = json.loads(man_file.read_text()) if man_file.exists() else {'runs': []}
    man['runs'] = [r for r in man['runs'] if r['prediction_file'] != pred_file.name] + [dict(
        prediction_file=pred_file.name, train_sha256=data['train_sha256'], val_sha256=data['val_sha256'],
        note='rows without a drug graph or outside the omics cohort were dropped before training')]
    man_file.write_text(json.dumps(man, indent=2))

    m = sex_metrics(pred, seed=seed)
    row = dict(split=split, model=model_name, fusion=spec['fusion'], family=spec['family'],
               layers='+'.join(spec['layers']), proteomics='proteomics' in spec['layers'], sex_config=spec['sex_name'],
               use_sex_input=bool(sex_cfg.get('use_sex_input')), use_xy_module=bool(sex_cfg.get('use_xy_module')),
               use_reweighting=bool(sex_cfg.get('use_reweighting')),
               shuffle_sex_labels=bool(sex_cfg.get('shuffle_sex_labels')), seed=seed, sex_labels=SEX_LABELS,
               n_train_rows=len(data['rows']['train']['y']), n_val_rows=len(data['rows']['val']['y']),
               best_epoch=info['best_epoch'], epochs_run=info['epochs_run'], val_rmse=info['best_val_rmse'],
               minutes=round((time.time() - t0) / 60, 2), **m)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not RESULTS_CSV.exists(), index=False)
    hdir = OUT_DIR / 'histories'; hdir.mkdir(exist_ok=True)
    pd.DataFrame(info['history']).to_csv(hdir / f'{model_name}__{split}__s{seed}.csv', index=False)
    print(f"{model_name:38s} {split} s{seed}  RMSE all {m['all_rmse']:.4f}  F {m['Female_rmse']:.4f}  "
          f"M {m['Male_rmse']:.4f}  gap {m['gap_F_minus_M']:+.4f} [{m.get('gap_ci_low', np.nan):+.3f}, "
          f"{m.get('gap_ci_high', np.nan):+.3f}]  ({row['minutes']} min)")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

### Sex-aware components (from `sex_aware/`, one config dict per run)
| flag | what it does | where |
|---|---|---|
| `use_sex_input` | appends one-hot `[female, male, unknown]` | fusion point (`concat_sex_aware`) |
| `use_xy_module` | small MLP over X/Y features → 32-d embedding | fusion point (`concat_sex_aware`) |
| `use_reweighting` | sex × tissue weights fitted on training lines (`reweighting.py`, within-tissue mode), used in `weighted_mse` | loss only |
| `shuffle_sex_labels` | control: permutes the labels the *model* sees within tissue; evaluation keeps true labels | inputs + weights |

`base` = all off (exact baseline); `sexaware` = all three on. `RUN_COMPONENT_ABLATIONS = True` adds each component alone and the shuffled-label control.

In [ ]:
OFF = dict(use_sex_input=False, use_xy_module=False, use_reweighting=False, shuffle_sex_labels=False)
ON = dict(use_sex_input=True, use_xy_module=True, use_reweighting=True, shuffle_sex_labels=False)
SEX_CONFIGS = {'base': OFF, 'sexaware': ON}
if RUN_COMPONENT_ABLATIONS:
    SEX_CONFIGS.update({
        'sexinput':  {**OFF, 'use_sex_input': True},
        'xymodule':  {**OFF, 'use_xy_module': True},
        'reweight':  {**OFF, 'use_reweighting': True},
        'sexaware_shuffled': {**ON, 'shuffle_sex_labels': True, 'shuffle_within_tissue': True},  # negative control
    })

## 3. Smoke test (run first)

In [ ]:
# Smoke test: tiny run of every model family, nothing saved. Should take a minute or two on a GPU.
_data = prepare_split(SPLITS_TO_RUN[0]); _T = to_tensors(_data)
display(_data['coverage'])
print('feature dims:', _data['dims'], '| XY features:', _data['xy'].shape[1])
for fam in FAMILIES:
    for sname in ('base', 'sexaware'):
        set_seed(0)
        cfg = {**BASE_SEX_CFG, **SEX_CONFIGS[sname]}
        labels, sex_t = make_sex_inputs(_data, cfg, 0)
        w = make_row_weights(_data, labels) if cfg['use_reweighting'] else None
        m = build_model(fam, SMOKE_LAYERS, _data, cfg).to(DEVICE)
        trip = dict(weight=MOLI_HP['triplet_weight'], margin=MOLI_HP['triplet_margin']) if fam == 'MOLI' else None
        info = train_model(m, _T, _data, sex_t, w, trip, cfg={**TRAIN, 'epochs': 1, 'max_train_rows': 5000}, verbose=False)
        print(f'{fam:8s} {sname:9s} params {sum(p.numel() for p in m.parameters()):,}  val RMSE after 1 epoch: {info["best_val_rmse"]:.3f}')
del _data, _T

## 4. Train: LF-DeepCDR and LF-MOLI × {no proteomics, + proteomics} × {base, sex-aware} × seeds

In [ ]:
for split in SPLITS_TO_RUN:
    data = prepare_split(split); T = to_tensors(data)
    print(f'===== {split} =====  dims {data["dims"]}'); display(data['coverage'])
    for seed in SEEDS:
        for fam in FAMILIES:
            for tag, layers in LF_LAYER_SETS.items():
                for sname, scfg in SEX_CONFIGS.items():
                    run_experiment(data, T, dict(fusion='LF', family=fam, layers=layers, layer_tag=tag,
                                                 sex_name=sname, sex_cfg=scfg), seed)
    del data, T

## 5. Results
Mean ± SD over seeds. `gap_F_minus_M` > 0 means higher error on female lines.

In [ ]:
res = pd.read_csv(RESULTS_CSV)
cols = ['all_rmse', 'Female_rmse', 'Male_rmse', 'gap_F_minus_M', 'all_pearson', 'Female_pearson', 'Male_pearson',
        'all_spearman']
summary = res.groupby(['split', 'model'])[cols].agg(['mean', 'std']).round(4)
summary.columns = [f'{a}_{b}' for a, b in summary.columns]
summary['n_seeds'] = res.groupby(['split', 'model']).size()
summary.to_csv(OUT_DIR / 'summary_by_model.csv')
summary

In [ ]:
for split, r in res.groupby('split'):
    p = r.groupby('model')[['Female_rmse', 'Male_rmse']].mean().sort_index()
    e = r.groupby('model')[['Female_rmse', 'Male_rmse']].std().reindex(p.index).fillna(0)
    ax = p.plot.barh(xerr=e, figsize=(8, 0.45 * len(p) + 1.5), color=['#c0392b', '#2471a3'])
    ax.set_xlim(p.values.min() * 0.97, p.values.max() * 1.02)
    ax.set_xlabel('test RMSE, ln IC50 (mean over seeds; lower is better)')
    ax.set_title(f'{split}: female vs male cell lines')
    plt.tight_layout(); plt.savefig(OUT_DIR / f'rmse_by_sex_{split}.png', dpi=150); plt.show()

In [ ]:
# 2 x 2 view per family: overall RMSE and |gap| for each (proteomics, sex-aware) cell, mean over seeds
r = res[res.sex_config.isin(['base', 'sexaware'])].assign(abs_gap=lambda d: d.gap_F_minus_M.abs())
for split, rs in r.groupby('split'):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, metric, title in zip(axes, ['all_rmse', 'abs_gap'], ['overall RMSE', '|female - male RMSE|']):
        t = rs.groupby(['family', 'proteomics', 'sex_config'])[metric].mean().unstack(['proteomics', 'sex_config'])
        t.plot.bar(ax=ax, rot=0, color=['#aab7b8', '#566573', '#a9cce3', '#1f618d'])
        ax.set_title(f'{split}: {title}'); ax.set_xlabel('')
        ax.legend([f"{'+prot' if p else 'no prot'}, {s}" for p, s in t.columns], fontsize=8)
        if metric == 'all_rmse':
            ax.set_ylim(t.values.min() * 0.97, t.values.max() * 1.02)
    plt.tight_layout(); plt.savefig(OUT_DIR / f'grid_{split}.png', dpi=150); plt.show()

In [ ]:
def paired_table(pairs):
    """pairs: list of (split, label_dict, model_a, model_b). Runs paired_compare per seed and pools."""
    rows = []
    for split, labels, a, b in pairs:
        for s in SEEDS:
            fa = OUT_DIR / 'predictions' / f'seed{s}' / f'{a}__{split}.csv'
            fb = OUT_DIR / 'predictions' / f'seed{s}' / f'{b}__{split}.csv'
            if not (fa.exists() and fb.exists()):
                continue
            rd = lambda f: pd.read_csv(f, dtype={'sample_id': str, 'cell_line_id': str, 'drug_id': str})
            rows.append(dict(split=split, seed=s, model_a=a, model_b=b, **labels,
                             **paired_compare(rd(fa), rd(fb), seed=s)))
    return pd.DataFrame(rows)

def pool_seeds(t, keys):
    keep = ['delta_rmse_all', 'delta_rmse_Female', 'delta_rmse_Male', 'delta_gap', 'delta_abs_gap']
    g = t.groupby(keys)
    out = g[keep].mean().round(4)
    out['n_seeds'] = g.size()
    out['seeds_B_better_overall'] = g['delta_rmse_all'].apply(lambda x: int((x < 0).sum()))
    out['seeds_CI_excludes_0_overall'] = g.apply(
        lambda d: int(((d.delta_rmse_all_ci_high < 0) | (d.delta_rmse_all_ci_low > 0)).sum()))
    out['seeds_gap_shrunk'] = g['delta_abs_gap'].apply(lambda x: int((x < 0).sum()))
    return out

### H3: proteomics vs no proteomics (paired)

In [ ]:
# H3: does adding proteomics lower error, and does it shrink the female-male gap?
# delta = (+proteomics) - (no proteomics), same sex-aware setting, identical test rows.
pairs = [(split, dict(family=f, sex_config=sn), f'LF_{f}_noprot_{sn}', f'LF_{f}_prot_{sn}')
         for split in SPLITS_TO_RUN for f in FAMILIES for sn in SEX_CONFIGS]
h3 = paired_table(pairs)
if len(h3):
    h3.to_csv(OUT_DIR / 'paired_proteomics_vs_noprot.csv', index=False)
    display(pool_seeds(h3, ['split', 'family', 'sex_config']))

### H4: sex-aware vs base (paired)

In [ ]:
# H4: does each sex-aware configuration reduce the gap without hurting overall RMSE, with and without proteomics?
pairs = [(split, dict(family=f, layer_tag=t, comparison=f'{sn} vs base'), f'LF_{f}_{t}_base', f'LF_{f}_{t}_{sn}')
         for split in SPLITS_TO_RUN for f in FAMILIES for t in LF_LAYER_SETS for sn in SEX_CONFIGS if sn != 'base']
h4 = paired_table(pairs)
if len(h4):
    h4.to_csv(OUT_DIR / 'paired_sexaware_vs_base.csv', index=False)
    display(pool_seeds(h4, ['split', 'family', 'layer_tag', 'comparison']))

In [ ]:
# Within-tissue female vs male RMSE (seed-averaged predictions), tissues with >= 3 lines of each sex in test
wt = []
for (split, model), _ in res.groupby(['split', 'model']):
    df = seed_mean_predictions(model, split)
    if df is not None:
        t = within_tissue(df)
        if len(t):
            wt.append(t.assign(split=split, model=model))
wt = pd.concat(wt, ignore_index=True) if wt else pd.DataFrame()
if len(wt):
    wt.to_csv(OUT_DIR / 'within_tissue_gap.csv', index=False)
    display(wt.pivot_table(index=['split', 'tissue'], columns='model', values='gap_F_minus_M').round(3))
else:
    print('No tissue has >= 3 lines of each sex in the test set for this split.')

## 6. Team evaluation harness

In [ ]:
# Official scoring with the team harness (per seed). Partial coverage (drugs without SMILES, cells without omics)
# needs --allow-partial; the harness then labels results EXPERIMENTAL/PARTIAL.
if RUN_HARNESS:
    for s in SEEDS:
        pdir = OUT_DIR / 'predictions' / f'seed{s}'
        if not any(pdir.glob('*__*.csv')):
            continue
        cmd = [sys.executable, str(REPO_DIR / 'harness_results' / 'evaluate.py'),
               '--splits-dir', str(SPLIT_ROOT), '--predictions-dir', str(pdir),
               '--output-dir', str(OUT_DIR / 'harness_results' / f'seed{s}'),
               '--model-runs', str(pdir / 'model_runs.json'), '--seed', str(s)]
        if SEX_LABEL_FILE is not None:
            cmd += ['--corrected-sex-labels', str(SEX_LABEL_FILE)]
        if ALL_ROWS.drug_id.isin(DRUG_POS).mean() < 1 or len(COHORT_IDS) < ALL_ROWS.cell_line_id.nunique():
            cmd.append('--allow-partial')
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(f'--- seed {s} (exit {r.returncode})'); print(r.stdout[-3000:]); print(r.stderr[-2000:])
    h = [pd.read_csv(f).assign(seed=int(f.parent.name[4:]))
         for f in (OUT_DIR / 'harness_results').glob('seed*/h2_performance_gap_table.csv')]
    if h:
        h = pd.concat(h, ignore_index=True)
        h.to_csv(OUT_DIR / 'harness_h2_all_seeds.csv', index=False)
        display(h[['seed', 'model', 'split_design', 'male_rmse', 'female_rmse', 'pooled_rmse_gap_female_minus_male',
                   'pooled_gap_ci_low', 'pooled_gap_ci_high', 'tissue_adjusted_rmse_gap_female_minus_male',
                   'within_tissue_permutation_p_two_sided', 'p_fdr_bh', 'analysis_status']].round(4))

## Notes / to discuss
* **Proteomics cohort.** All runs are restricted to lines with ≥50% of proteins measured, so H3 compares identical observations. The cost is that this notebook's cohort can be smaller than a no-proteomics-only analysis could use; the coverage table above shows the counts.
* **MOLI adaptation.** Original MOLI is drug-specific binary classification. Here it's pan-drug regression: shared GCN drug encoder, MLP head (a linear head can't model cell×drug interaction), MSE + 0.2 × triplet loss, where "sensitive" = ln IC50 below that drug's training median.
* **DeepCDR adaptation.** CNV replaces methylation; the mutation CNN's kernel and stride are scaled for ~1k gene-level features; drug graphs are rebuilt from SMILES for all GDSC2 drugs (DeepCDR's files cover 70).
* **Optimiser.** Both families use AdamW with the same schedule so the comparison is controlled.
* **Expression input** is Harmonizome association scores, not TPM (Week 2 README). Supplying `XY_CSV` from `export_xy.py` gives the XY module real DepMap expression.
* **Drug coverage.** Check `drug_table.csv`; add missing SMILES to `drug_smiles_overrides.csv`. If coverage changes, delete `results.csv` and `predictions/` so every run uses the same rows.
* Cell lines grow without sex hormones, so any gap reflects cell-intrinsic differences; findings are hypotheses, not treatment guidance.